# Validation of the Control Behavior for both ```OLTC``` and ```FSM``` Module Control
Plotting of the simulation results, saving them and comparing to PowerFactory Data.

# Imports and Predefinitions

In [ ]:
# %matplotlib inline
# import ipympl

import os
import sys

parent = os.path.abspath("/Users/maxikoehler/Documents/GitHub/diffpssi-ma-kohler/")
sys.path.insert(1, parent)
sys.path.append(
    str(os.path.dirname(os.path.dirname(os.path.abspath("oltc_validation.ipynb"))))
)

import matplotlib.pyplot as plt
import matplotlib as mpl

from diffpssi.tools import *

import numpy as np
import pandas as pd

import diffpssi.grid_library.ibb_trans_model as mdl
from diffpssi.power_sim_lib.simulator import PowerSystemSimulation as Pss
from diffpssi.power_sim_lib.simulator import Recorder
# from diffpssi.tools.calcs import *

In [ ]:
def load_smib(control="oltc", p_load=100, q_load=0, param_dict_oltc=None):
    """
    Loads the grid data of the IBB transformer model.
    Returns: The grid data in form of a dictionary.

    """
    return {
        "base_mva": 2200,
        "f": 60,
        "slack_bus": "Bus 0",
        "base_voltage": 100,
        "busses": [
            ["name", "V_n"],
            ["Bus 0", 100],
            ["Bus 1", 100],
            ["Bus 2", 10],
        ],
        "lines": [
            ["name", "from_bus", "to_bus", "length", "unit", "R", "X", "B"],
            ["Line 1", "Bus 0", "Bus 1", 1, "Ohm", 0, 0.11, 0],  # before: 0.0484 p.u.
        ],
        "transformers": [
            [
                "type",
                "control",
                "name",
                "from_bus",
                "to_bus",
                "S_n",
                "tap_side",
                "measure_side",
                "V_n_from",
                "V_n_to",
                "R",
                "X",
                "param_dict_oltc",
            ],
            [
                "oltc",
                control,
                "T1",
                "Bus 1",
                "Bus 2",
                4400,
                "hv",
                "hv",
                100,
                10,
                0,
                0.15,
                param_dict_oltc,
            ],
        ],
        "generators": {
            "GEN": [
                [
                    "name",
                    "bus",
                    "S_n",
                    "V_n",
                    "P",
                    "V",
                    "H",
                    "D",
                    "X_d",
                    "X_q",
                    "X_d_t",
                    "X_q_t",
                    "X_d_st",
                    "X_q_st",
                    "T_d0_t",
                    "T_q0_t",
                    "T_d0_st",
                    "T_q0_st",
                ],
                [
                    "G1",
                    "Bus 0",
                    11000,
                    100,
                    -1998,
                    0.995,
                    3.5e7,
                    0,
                    1.81,
                    1.76,
                    0.3,
                    0.65,
                    0.23,
                    0.23,
                    8.0,
                    1,
                    0.03,
                    0.07,
                ],
                [
                    "G2",
                    "Bus 2",
                    2200,
                    10,
                    1998,
                    1,
                    3.5,
                    0,
                    1.81,
                    1.76,
                    0.3,
                    0.65,
                    0.23,
                    0.23,
                    8.0,
                    1,
                    0.03,
                    0.07,
                ],
            ],
        },
        "loads": {
            "ZIP": [
                ["name", "bus", "P", "Q", "model"],
                ["L1", "Bus 1", p_load, q_load, "Z"],
            ],
        },
    }

In [ ]:
def load_sm_load(control="oltc", p_load=400, q_load=0, param_dict_oltc=None):
    """
    Loads the grid data of the IBB transformer model.
    Returns: The grid data in form of a dictionary.

    """
    return {
        "base_mva": 2200,
        "f": 60,
        "slack_bus": "Bus 0",
        "base_voltage": 100,
        "busses": [
            ["name", "V_n"],
            ["Bus 0", 10],
            ["Bus 1", 100],
        ],
        "transformers": [
            [
                "type",
                "control",
                "name",
                "from_bus",
                "to_bus",
                "S_n",
                "tap_side",
                "measure_side",
                "V_n_from",
                "V_n_to",
                "R",
                "X",
                "param_dict_oltc",
            ],
            [
                "oltc",
                control,
                "T1",
                "Bus 0",
                "Bus 1",
                2200,
                "hv",
                "hv",
                10,
                100,
                0,
                0.15,
                param_dict_oltc,
            ],
        ],
        "generators": {
            "GEN": [
                [
                    "name",
                    "bus",
                    "S_n",
                    "V_n",
                    "P",
                    "V",
                    "H",
                    "D",
                    "X_d",
                    "X_q",
                    "X_d_t",
                    "X_q_t",
                    "X_d_st",
                    "X_q_st",
                    "T_d0_t",
                    "T_q0_t",
                    "T_d0_st",
                    "T_q0_st",
                ],
                [
                    "G1",
                    "Bus 0",
                    2200,
                    10,
                    -1998,
                    1,
                    3.5,
                    0,
                    1.81,
                    1.76,
                    0.3,
                    0.65,
                    0.23,
                    0.23,
                    8.0,
                    1,
                    0.03,
                    0.07,
                ],
            ],
        },
        "loads": {
            "ZIP": [
                ["name", "bus", "P", "Q", "model"],
                ["L1", "Bus 1", p_load, q_load, "Z"],
            ],
        },
    }

# OLTC Module

## 1 OLTC in SMIB with Load

In [ ]:
def record_dict_smib(simulation, call=False):
    record_dict = {
        "Bus 0: Voltage Magnitude": simulation.busses[0].get_value("voltage_mag"),
        "Bus 1: Voltage Magnitude": simulation.busses[1].get_value("voltage_mag"),
        "Bus 2: Voltage Magnitude": simulation.busses[2].get_value("voltage_mag"),
        r"Transformer $u_\mathrm{l}$": simulation.trafos[0].oltc.u_l,
        r"Transformer $v_\mathrm{dead}$": simulation.trafos[0].oltc.v_dead,
        r"Integrator $m$": simulation.trafos[0].oltc.integ,
    }
    if call:
        return record_dict.values()
    else:
        return record_dict

### 1.1 Sim Setup and Run

In [ ]:
param_dict_oltc = {
    "t_1": 5,
    "db": 0.05,
    "delta_m": 0.02,
    "m_max": 1.1,
    "m_min": 0.9,
    "v_ref": 1,
}

parallel_sims = 1
sim = Pss(
    parallel_sims=parallel_sims,
    sim_time=120,
    time_step=0.005,
    solver="heun",
    grid_data=load_smib(param_dict_oltc=param_dict_oltc),
)

sim.trafos[0].oltc.dir = -1

sim.add_param_event(1, sim.busses[1].models[0], "p_soll_mw", 1100)

rec = Recorder(sim=sim, recorder_dict=record_dict_smib)

sim.set_record_function(rec.record_fun)
record_list = rec.record_list()

t, recorder = sim.run()

### 1.2 Some details of whats going on

In [ ]:
plt.figure()

plt.plot(t, recorder[0, :, -1], label=record_list[-1])
plt.grid()
plt.ylabel(record_list[-1])
plt.xlabel("Time in s")
plt.savefig("./data/oltc_ex-smib_integrator.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(t, recorder[0, :, -3], label=record_list[-3])
ax[0].grid()
ax[0].set_title(r"$u_\mathrm{l}$")
ax[1].plot(t, recorder[0, :, -2], label=record_list[-2])
ax[1].grid()
ax[1].set_title(r"$v_\mathrm{dead}$")
ax[1].set_ylabel("Voltage Magnitude in p.u.")

# fig.supylabel('Voltage Magnitude in p.u.')
fig.supxlabel("Time in s")

plt.savefig("./data/oltc_ex-smib_signals.pdf")
plt.show()

In [ ]:
plt.figure()

plt.plot(t, recorder[0, :, -3], label=record_list[-3])
plt.grid()
plt.legend()
plt.ylabel("Voltage Magnitude in p.u.")
plt.xlabel("Time in s")

plt.savefig("./data/oltc_ex-smib_ul.pdf")

plt.show()

### 1.3 Saving data, generation and saving of plots 

#### Only the Simulation Data from ```diffpssi```

In [ ]:
plt.figure(figsize=(10, 6))
for i in range(3):
    plt.plot(t, recorder[0, :, i], label=record_list[i])
plt.grid()
plt.hlines(
    sim.trafos[0].oltc.db + sim.trafos[0].oltc.v_ref,
    t[0],
    t[-1],
    colors=ees_red,
    label="Deadband",
    linestyle="--",
)
plt.legend()
# plt.ylim(0.9, 1.15)
plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")

plt.savefig("./data/tds_oltc_ex-smib.pdf")
plt.show()

In [ ]:
data = pd.DataFrame(recorder[0, :, :])
data.columns = record_list
data["t"] = t
data.set_index("t", inplace=True)

# data.to_csv('./data/validation/oltc/PY_smib_w_Zload100-700_oltc-validation.csv')

data.head()

#### Loading the Data from PowerFactory

In [ ]:
data_pf_oltc_smib = pd.read_csv(
    "./data/validation_oltc/smib_w_Zload100-700_oltc-validation_HV.csv",
    sep=";",
    dtype=float,
)
data_pf_oltc_smib.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_oltc_smib.set_index("t", inplace=True)

data_pf_oltc_smib.head()

#### Comparing the Data: ```diffpssi``` vs. PowerFactory

In [ ]:
time = data.index

# Calculate mean error
mean_errors_oltc_smib = np.zeros((3, len(time)))
for i in range(3):
    ext_voltages = data_pf_oltc_smib.iloc[:, i].values
    model_voltage = data.iloc[:, i].values
    errors_oltc_smib = [
        abs(ext_v - model_v) for model_v, ext_v in zip(model_voltage, ext_voltages)
    ]
    mean_errors_oltc_smib[i] = errors_oltc_smib

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 8))  # , sharex=True, sharey=True)

for i in range(3):
    row, col = divmod(i, 2)
    axs[row, col].plot(
        data_pf_oltc_smib.index.values,
        data_pf_oltc_smib.iloc[:, i].values,
        label="PowerFactory Data",
    )
    axs[row, col].plot(
        time, data.iloc[:, i].values, linestyle="-", label="diffpssi Data"
    )
    axs[row, col].set_title(f"Voltage Comparison for Bus {i}")
    # axs[row, col].set_xlabel('Time (s)')
    # axs[row, col].set_ylabel('Voltage in p.u.')
    axs[row, col].legend()
    axs[row, col].grid(True)

# Add mean error plot
axs[1, 1].plot(time, mean_errors_oltc_smib[0], label="Error for Bus 0")
axs[1, 1].plot(time, mean_errors_oltc_smib[1], label="Error for Bus 1")
axs[1, 1].plot(time, mean_errors_oltc_smib[2], label="Error for Bus 2")
# axs[1, 1].set_ylabel˚('Error in p.u.')
axs[1, 1].set_title("Absolute Error Comparison")
# axs[1, 1].set_xlabel('Time (s)')
axs[1, 1].legend()
axs[1, 1].grid(True)

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")

plt.tight_layout()
plt.savefig("./data/comp_oltc_ext-smib.pdf")

plt.show()

## 2 OLTC with Machine and Load only

In [ ]:
def record_dict_smload(simulation, call=False):
    record_dict = {
        "Bus 0: voltage magnitude": simulation.busses[0].get_value("voltage_mag"),
        "Bus 1: voltage magnitude": simulation.busses[1].get_value("voltage_mag"),
        r"Transformer $u_\mathrm{l}$": simulation.trafos[0].oltc.u_l,
    }
    if call:
        return record_dict.values()
    else:
        return record_dict

### 2.1 Sim Set-Up and Run

In [ ]:
param_dict_oltc = {
    "t_1": 5,
    "db": 0.05,
    "delta_m": 0.02,
    "m_max": 1.1,
    "m_min": 0.9,
    "v_ref": 1,
}

parallel_sims = 1
sim_2 = Pss(
    parallel_sims=parallel_sims,
    sim_time=120,
    time_step=0.005,
    solver="heun",
    grid_data=load_sm_load(param_dict_oltc=param_dict_oltc),
)

sim_2.trafos[0].oltc.dir = -1

sim_2.add_param_event(1, sim_2.busses[1].models[0], "p_soll_mw", 800)

rec_2 = Recorder(sim=sim_2, recorder_dict=record_dict_smload)

sim_2.set_record_function(rec_2.record_fun)
record_list_2 = rec_2.record_list()

t, recorder_2 = sim_2.run()

### 2.2 Some Details of Whats going on

In [ ]:
plt.figure()

plt.plot(t, recorder_2[0, :, -1], label=record_list_2[-1])
plt.grid()
plt.ylabel(record_list_2[-1])
plt.xlabel("Time in s")

plt.show()

### 2.3 Saving Data, Generation and Saving of Plots

#### Only the data from ```diffpssi```

In [ ]:
plt.figure(figsize=(10, 6))
for i in range(2):
    plt.plot(t, recorder_2[0, :, i], label=record_list_2[i])
plt.grid()
plt.legend()
# plt.ylim(0.9, 1.15)
plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")

plt.savefig("./data/tds_oltc_simple-load.pdf")

plt.show()

In [ ]:
data_sm_load = pd.DataFrame(recorder_2[0, :, :])
data_sm_load.columns = record_list_2
data_sm_load["t"] = t
data_sm_load.set_index("t", inplace=True)

# data.to_csv('./data/validation/oltc/PY_smib_w_Zload100-700_oltc-validation.csv')

data_sm_load.head()

#### Loading from ```PowerFactory```

In [ ]:
data_pf_oltc_smload = pd.read_csv(
    "./data/validation_oltc/sm_w_Zload400-800_oltc-validation_HV.csv",
    sep=";",
    dtype=float,
)
data_pf_oltc_smload.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
    },
    inplace=True,
)
data_pf_oltc_smload.set_index("t", inplace=True)

data_pf_oltc_smload.head()

In [ ]:
data_pf_oltc_smload_lv = pd.read_csv(
    "./data/validation_oltc/sm_w_Zload400-800_oltc-validation_LV.csv",
    sep=";",
    dtype=float,
)
data_pf_oltc_smload_lv.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
    },
    inplace=True,
)
data_pf_oltc_smload_lv.set_index("t", inplace=True)

data_pf_oltc_smload_lv.head()

#### Comparing Against Each Other

In [ ]:
time = data_sm_load.index

# Calculate mean error
mean_errors_oltc_simple = np.zeros((2, len(time)))
for i in range(2):
    ext_voltages = data_pf_oltc_smload.iloc[:, i].values
    model_voltage = data_sm_load.iloc[:, i].values
    errors_oltc_simple = [
        abs(ext_v - model_v) for model_v, ext_v in zip(model_voltage, ext_voltages)
    ]
    mean_errors_oltc_simple[i] = errors_oltc_simple

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

for i in range(2):
    axs[i].plot(
        data_pf_oltc_smload.index.values,
        data_pf_oltc_smload.iloc[:, i].values,
        color=ees_red,
        label="PowerFactory Data",
    )
    axs[i].plot(
        time, data_sm_load.iloc[:, i].values, linestyle="dashed", label="diffpssi Data"
    )
    axs[i].set_title(f"Bus {i}")
    # axs[i].set_xlabel('Time (s)')
    # axs[i].set_ylabel('Voltage in p.u.')
    axs[i].legend()
    axs[i].grid(True)

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")

plt.tight_layout()
# plt.savefig('./data/comp_oltc_simple-load.pdf')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

for i in range(2):
    axs[i].plot(
        data_pf_oltc_smload_lv.index.values,
        data_pf_oltc_smload_lv.iloc[:, i].values,
        label="PowerFactory Data",
    )
    axs[i].plot(
        time, data_sm_load.iloc[:, i].values, linestyle="-", label="diffpssi Data"
    )
    axs[i].set_title(f"Bus {i}")
    # axs[i].set_xlabel('Time (s)')
    # axs[i].set_ylabel('Voltage in p.u.')
    axs[i].legend()
    axs[i].grid(True)

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")

plt.tight_layout()
plt.savefig("./data/comp_oltc_simple-load_discrept.pdf")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

# Add mean error plot
plt.plot(time, mean_errors_oltc_simple[0], label="Absolute Error for Bus 0")
plt.plot(time, mean_errors_oltc_simple[1], label="Absolute Error for Bus 1")
# plt.title('Absolute Error Comparison')
plt.ylabel("Voltage Magnitude in p.u.")
plt.xlabel("Time in s")
plt.legend()
plt.grid(True)

plt.savefig("./data/error_oltc_simple-load.pdf")

plt.show()

# FSM Module

## 1 FSM in SMIB with Load (Dependent on Voltage Difference)

In [ ]:
def record_dict_smib_fsm(simulation, call=False):
    record_dict = {
        "Bus 0: Voltage Magnitude": simulation.busses[0].get_value("voltage_mag"),
        "Bus 1: Voltage Magnitude": simulation.busses[1].get_value("voltage_mag"),
        "Bus 2: Voltage Magnitude": simulation.busses[2].get_value("voltage_mag"),
        r"Transformer Ratio $\vartheta$": simulation.trafos[0].oltc.u_l,
        r"Transformer FSM Position $k$": simulation.trafos[0].oltc.tap_pos_k,
        r"Transformer OLTC Position $m$": simulation.trafos[0].oltc.tap_pos_m,
        r"Voltage Difference $v_\mathrm{diff}$": simulation.trafos[0].oltc.v_diff,
        # r'Voltage difference $v_\mathrm{dead}$':    simulation.trafos[0].oltc.v_dead,
        r"Integrator $k$": simulation.trafos[0].oltc.integ_k,
        r"Integrator $m$": simulation.trafos[0].oltc.integ_m,
    }
    if call:
        return record_dict.values()
    else:
        return record_dict

### 1.1 Sim Set Up and Run

In [ ]:
param_dict_fsm = {
    "t_m": 0.02,
    "t_k": 5,
    "db": 0.025,
    "delta_m": 2,
    "delta_k": 0.02,
    "m_max": 4,
    "m_min": -4,
    "k_max": 10,
    "k_min": -10,
    "v_ref": 1,
    "gamma_max": 8,
    "pt_1": 0.01,
}

parallel_sims = 1
sim_3 = Pss(
    parallel_sims=parallel_sims,
    sim_time=120,
    time_step=0.001,
    solver="heun",
    grid_data=load_smib(control="fsm", param_dict_oltc=param_dict_fsm),
)

sim_3.trafos[0].oltc.dir = -1

# sim_3.add_sc_event(1, 1.05, 'Bus 1')
sim_3.add_param_event(1, sim_3.busses[1].models[0], "p_soll_mw", 700)

rec_3 = Recorder(sim=sim_3, recorder_dict=record_dict_smib_fsm)

sim_3.set_record_function(rec_3.record_fun)
record_list_3 = rec_3.record_list()

t, recorder_3 = sim_3.run()

### 1.2 Some Details of What's Going On

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axs[0].plot(t, recorder_3[0, :, 3], label=record_list_3[3])
axs[0].grid()
axs[0].legend()

for i in range(4, 6):
    axs[1].plot(t, recorder_3[0, :, i], label=record_list_3[i])
axs[1].grid()
axs[1].legend()

for i in range(6, 8):
    axs[2].plot(t, recorder_3[0, :, i], label=record_list_3[i])
axs[2].grid()
axs[2].legend()
axs[2].set_xlabel("Time [s]")
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axs[0].plot(t, recorder_3[0, :, 3], label=record_list_3[3])
axs[0].grid()
axs[0].legend()

for i in range(4, 6):
    axs[1].plot(t, recorder_3[0, :, i], label=record_list_3[i])
axs[1].grid()
axs[1].legend()

fig.supxlabel("Time in s")

plt.savefig("./data/signals_ratio-k-m_fsm_vdiff.pdf")

plt.show()

### 1.3 Saving Data, Generation and Savong of Plots

#### Only Simulation Data from ```diffpssi```

In [ ]:
plt.figure(figsize=(10, 6))
for i in range(3):
    plt.plot(t, recorder_3[0, :, i], label=record_list_3[i])
plt.hlines(
    sim_3.trafos[0].oltc.db + sim_3.trafos[0].oltc.v_ref,
    t[0],
    t[-1],
    colors=ees_red,
    label=r"$v_\mathrm{dead}$",
)
plt.hlines(
    2 * sim_3.trafos[0].oltc.db + sim_3.trafos[0].oltc.v_ref,
    t[0],
    t[-1],
    colors=ees_red,
    linestyles="dashed",
    label=r"$2 \cdot v_\mathrm{dead}$",
)
plt.grid()
plt.legend()
# plt.ylim(0.9, 1.15)
plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")

plt.savefig("./data/tds_fsm_vdiff.pdf")

plt.show()

In [ ]:
data_smib_fsm_1 = pd.DataFrame(recorder_3[0, :, :])
data_smib_fsm_1.columns = record_list_3
data_smib_fsm_1["t"] = t
data_smib_fsm_1.set_index("t", inplace=True)

# data.to_csv('./data/validation/oltc/PY_smib_w_Zload100-700_oltc-validation.csv')

data_smib_fsm_1.head()

#### Loading Data from ```PowerFactory```

In [ ]:
data_pf_fsm_smib_1 = pd.read_csv(
    "./data/validation_oltc/smib_w_Zload100-700_fsm-validation_HV.csv",
    sep=";",
    dtype=float,
)
data_pf_fsm_smib_1.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_fsm_smib_1.set_index("t", inplace=True)


data_pf_fsm_smib_1.head()

#### Comparing Against each Other

In [ ]:
time = data_smib_fsm_1.index

# Calculate mean error
mean_errors_smib_fsm_1 = np.zeros((3, len(time)))
for i in range(3):
    ext_voltages = data_pf_fsm_smib_1.iloc[:, i].values
    model_voltage = np.abs(data_smib_fsm_1.iloc[:, i].values)
    errors_smib_fsm_1 = []
    for model_v, ext_v in zip(model_voltage, ext_voltages):
        errors_smib_fsm_1.append(abs(ext_v - model_v))
    mean_errors_smib_fsm_1[i] = errors_smib_fsm_1

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 8))

for i in range(3):
    row, col = divmod(i, 2)
    axs[row, col].plot(
        data_pf_fsm_smib_1.index.values,
        data_pf_fsm_smib_1.iloc[:, i].values,
        label="PowerFactory Data",
    )
    axs[row, col].plot(
        time, data_smib_fsm_1.iloc[:, i].values, linestyle="-", label="diffpssi Data"
    )
    axs[row, col].set_title(f"Voltage Comparison for Bus {i}")
    # axs[row, col].set_xlabel('Time (s)')
    # axs[row, col].set_ylabel('Voltage in p.u.')
    axs[row, col].legend()
    axs[row, col].grid(True)

# Add mean error plot
axs[1, 1].plot(time, mean_errors_smib_fsm_1[0], label="Error for Bus 0")
axs[1, 1].plot(time, mean_errors_smib_fsm_1[1], label="Error for Bus 1")
axs[1, 1].plot(time, mean_errors_smib_fsm_1[2], label="Error for Bus 2")
axs[1, 1].set_title("Absolute Error Comparison")
# axs[1, 1].set_ylabel('Error in p.u.')
# axs[1, 1].set_xlabel('Time (s)')
axs[1, 1].legend()
axs[1, 1].grid(True)

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")

plt.tight_layout()
plt.savefig("./data/error_comp_fsm_vdiff.pdf")
plt.show()

In [ ]:
plt.figure()

plt.plot(time, mean_errors_smib_fsm_1[0], label="Error for Bus 0")
plt.plot(time, mean_errors_smib_fsm_1[1], label="Error for Bus 1")
plt.plot(time, mean_errors_smib_fsm_1[2], label="Error for Bus 2")
plt.legend()
plt.grid(True)

plt.show()

## 2 FSM in SMIB with Load (FSM Preferred)

In [ ]:
def record_dict_smib_fsm(simulation, call=False):
    record_dict = {
        "Bus 0: Voltage Magnitude": simulation.busses[0].get_value("voltage_mag"),
        "Bus 1: Voltage Magnitude": simulation.busses[1].get_value("voltage_mag"),
        "Bus 2: Voltage Magnitude": simulation.busses[2].get_value("voltage_mag"),
        r"Transformer Ratio $\vartheta$": simulation.trafos[0].oltc.u_l,
        r"Transformer FSM Position $k$": simulation.trafos[0].oltc.tap_pos_k,
        r"Transformer OLTC Position $m$": simulation.trafos[0].oltc.tap_pos_m,
        r"Voltage Difference $v_\mathrm{diff}$": simulation.trafos[0].oltc.v_diff,
        # r'Voltage difference $v_\mathrm{dead}$':    simulation.trafos[0].oltc.v_dead,
        r"Integrator $k$": simulation.trafos[0].oltc.integ_k,
        r"Integrator $m$": simulation.trafos[0].oltc.integ_m,
    }
    if call:
        return record_dict.values()
    else:
        return record_dict

### 2.1 Sim Set Up and Run

In [ ]:
param_dict_fsm = {
    "t_m": 0.02,
    "t_k": 5,
    "db": 0.025,
    "delta_m": 2,
    "delta_k": 0.02,
    "m_max": 4,
    "m_min": -4,
    "k_max": 10,
    "k_min": -10,
    "v_ref": 1,
    "gamma_max": 8,
    "pt_1": 0.01,
}

parallel_sims = 1
sim_5 = Pss(
    parallel_sims=parallel_sims,
    sim_time=120,
    time_step=0.001,
    solver="heun",
    grid_data=load_smib(control="fsm_2", param_dict_oltc=param_dict_fsm),
)

sim_5.trafos[0].oltc.dir = -1

# sim_5.add_sc_event(1, 1.05, 'Bus 1')
sim_5.add_param_event(1, sim_5.busses[1].models[0], "p_soll_mw", 1100)

rec_5 = Recorder(sim=sim_5, recorder_dict=record_dict_smib_fsm)

sim_5.set_record_function(rec_5.record_fun)
record_list_5 = rec_5.record_list()

t_5, recorder_5 = sim_5.run()

### 2.2 Some Details of What's Going On

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axs[0].plot(t, recorder_5[0, :, 3], label=record_list_5[3])
axs[0].grid()
axs[0].legend()

for i in range(4, 6):
    axs[1].plot(t, recorder_5[0, :, i], label=record_list_5[i])
axs[1].grid()
axs[1].legend()

for i in range(6, 8):
    axs[2].plot(t, recorder_5[0, :, i], label=record_list_5[i])
axs[2].grid()
axs[2].legend()
axs[2].set_xlabel("Time [s]")
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

time_plot = np.arange(0, 120.005, 0.005)

axs[0].plot(t, recorder_5[0, :, 3], label=record_list_5[3])
axs[0].grid()
axs[0].legend()

for i in range(4, 6):
    axs[1].plot(t, recorder_5[0, :, i], label=record_list_5[i])
axs[1].grid()
axs[1].legend()

fig.supxlabel("Time in s")

plt.savefig("./data/signals_ratio-k-m_fsm_pref.pdf")

plt.show()

In [ ]:
plt.figure()

plt.plot(t, recorder_5[0, :, -1], label=record_list_5[-1])
plt.grid()
plt.legend()

plt.show()

In [ ]:
plt.figure()

plt.plot(t, recorder_5[0, :, -2], label=record_list_5[-2])
plt.grid()
plt.legend()

plt.show()

### 2.3 Saving Data, Generation and Savong of Plots

#### Only Simulation Data from ```diffpssi```

In [ ]:
plt.figure(figsize=(10, 6))
for i in range(3):
    plt.plot(t, recorder_5[0, :, i], label=record_list_5[i])
plt.hlines(
    sim_5.trafos[0].oltc.db + sim_5.trafos[0].oltc.v_ref,
    t[0],
    t[-1],
    colors=ees_red,
    label=r"$v_\mathrm{dead}$",
)
plt.hlines(
    2 * sim_5.trafos[0].oltc.db + sim_5.trafos[0].oltc.v_ref,
    t[0],
    t[-1],
    colors=ees_red,
    linestyles="dashed",
    label=r"$\Delta m \cdot v_\mathrm{dead}$",
)
plt.grid()
plt.legend()
# plt.ylim(0.9, 1.15)
plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")
plt.savefig("./data/tds_fsm_preferred.pdf")
plt.show()

In [ ]:
data_smib_fsm_pref = pd.DataFrame(recorder_5[0, :, :])
data_smib_fsm_pref.columns = record_list_5
data_smib_fsm_pref["t"] = t
data_smib_fsm_pref.set_index("t", inplace=True)

# data.to_csv('./data/validation/oltc/PY_smib_w_Zload100-700_oltc-validation.csv')

data_smib_fsm_pref.head()

#### Loading Data from ```PowerFactory```

In [ ]:
data_pf_fsm_smib = pd.read_csv(
    "./data/validation_oltc/smib_w_Zload100-700_fsm-validation_HV.csv",
    sep=";",
    dtype=float,
)
data_pf_fsm_smib.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
        "Voltage, Magnitude in p.u..2": "Bus 2: voltage magnitude",
    },
    inplace=True,
)
data_pf_fsm_smib.set_index("t", inplace=True)

data_pf_fsm_smib.head()

#### Comparing Against each Other

In [ ]:
time = data_smib_fsm_pref.index

# Calculate mean error
mean_errors_smib_fsm_2 = np.zeros((3, len(time)))
for i in range(3):
    ext_voltages = data_pf_fsm_smib.iloc[:, i].values
    model_voltage = data_smib_fsm_pref.iloc[:, i].values
    errors_smib_fsm = [
        abs(ext_v - model_v) for model_v, ext_v in zip(model_voltage, ext_voltages)
    ]
    mean_errors_smib_fsm_2[i] = errors_smib_fsm

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 8))

for i in range(3):
    row, col = divmod(i, 2)
    axs[row, col].plot(
        data_pf_fsm_smib.index.values,
        data_pf_fsm_smib.iloc[:, i].values,
        label="PowerFactory Data",
    )
    axs[row, col].plot(
        time, data_smib_fsm_pref.iloc[:, i].values, linestyle="-", label="diffpssi Data"
    )
    axs[row, col].set_title(f"Voltage Comparison for Bus {i}")
    # axs[row, col].set_xlabel('Time (s)')
    # axs[row, col].set_ylabel('Voltage in p.u.')
    axs[row, col].legend()
    axs[row, col].grid(True)

# Add mean error plot
axs[1, 1].plot(time, mean_errors_smib_fsm_2[0], label="Error for Bus 0")
axs[1, 1].plot(time, mean_errors_smib_fsm_2[1], label="Error for Bus 1")
axs[1, 1].plot(time, mean_errors_smib_fsm_2[2], label="Error for Bus 2")
axs[1, 1].set_title("Absolute Error Comparison")
# axs[1, 1].set_ylabel('Error in p.u.')
# axs[1, 1].set_xlabel('Time (s)')
axs[1, 1].legend()
axs[1, 1].grid(True)

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")

plt.tight_layout()
# plt.savefig('./examples/master_thesis/ibb_transformer/data/comp_simple_pi.pdf')
plt.show()

## 3 FSM with Machine with Load only

In [ ]:
def record_dict_smload_fsm(simulation, call=False):
    record_dict = {
        "Bus 0: Voltage Magnitude": simulation.busses[0].get_value("voltage_mag"),
        "Bus 1: Voltage Magnitude": simulation.busses[1].get_value("voltage_mag"),
        "Voltage Measurement": simulation.trafos[0].oltc.v_measure,
        # r'Transformer Ratio $\vartheta$':         simulation.trafos[0].oltc.u_l,
        # r'Transformer FSM Position $k$':               simulation.trafos[0].oltc.tap_pos_k,
        # r'Transformer OLTC Position $m$':              simulation.trafos[0].oltc.tap_pos_m,
        # r'Voltage Difference $v_\mathrm{diff}$':    simulation.trafos[0].oltc.v_diff,
        # r'Integrator $k$':                          simulation.trafos[0].oltc.integrator_k.state_1,
        # r'Integrator $m$':                          simulation.trafos[0].oltc.integrator_m.state_1,
        # r'$v_\mathrm{dead}$':                       simulation.trafos[0].oltc.v_dead,
    }
    if call:
        return record_dict.values()
    else:
        return record_dict

### 3.1 Sim Set Up and Run

In [ ]:
param_dict_fsm = {
    "t_m": 0.02,
    "t_k": 5,
    "db": 0.025,
    "delta_m": 2,
    "delta_k": 0.02,
    "m_max": 4,
    "m_min": -4,
    "k_max": 10,
    "k_min": -10,
    "v_ref": 1,
    "gamma_max": 8,
    "pt_1": 0,  # 0.001,
}

parallel_sims = 1
sim = Pss(
    parallel_sims=parallel_sims,
    sim_time=120,
    time_step=0.005,
    solver="heun",
    grid_data=load_sm_load(control="fsm_2", param_dict_oltc=param_dict_fsm),
)

# sim.trafos[0].from_bus = 'hv'
sim.trafos[0].oltc.dir = -1

sim.add_param_event(1, sim.busses[1].models[0], "p_soll_mw", 800)

rec_4 = Recorder(sim=sim, recorder_dict=record_dict_smload_fsm)

sim.set_record_function(rec_4.record_fun)
record_list_4 = rec_4.record_list()

t, recorder_4 = sim.run()

### 3.2 Some Details of What's Going On?

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

for i in range(8, 9):
    axs[0].plot(t, recorder_4[0, :, i], label=record_list_4[i])
axs[0].grid()
axs[0].legend()

for i in range(7, 8):
    axs[1].plot(t, recorder_4[0, :, i], label=record_list_4[i])
axs[1].grid()
axs[1].legend()
axs[1].set_xlabel("Time in s")
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axs[0].plot(t, recorder_4[0, :, 3], label=record_list_4[3])
axs[0].grid()
axs[0].legend()

for i in range(4, 6):
    axs[1].plot(t, recorder_4[0, :, i], label=record_list_4[i])
axs[1].grid()
axs[1].legend()

fig.supxlabel("Time in s")

plt.savefig("./data/signals_ratio-k-m_fsm_simple.pdf")

plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(t, recorder_4[0, :, -1], label=record_list_4[-1])

plt.legend()
plt.grid()
plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")

plt.savefig("./data/signal_vdead_fsm_simple.pdf")

plt.show()

### 3.3 Saving Data, Generation and Saving of Plots

#### Only the Data from ```diffpssi```

In [ ]:
plt.figure(figsize=(10, 6))
for i in range(2):
    plt.plot(t, recorder_4[0, :, i], label=record_list_4[i])
plt.grid()
plt.legend()
# plt.ylim(0.9, 1.15)
plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")
plt.savefig("./data/tds_fsm_simple.pdf")
plt.show()

In [ ]:
data_sm_load_fsm = pd.DataFrame(recorder_4[0, :, :])
data_sm_load_fsm.columns = record_list_4
data_sm_load_fsm["t"] = t
data_sm_load_fsm.set_index("t", inplace=True)

# data.to_csv('./data/validation/oltc/PY_smib_w_Zload100-700_fsm-validation.csv')

data_sm_load_fsm.head()

#### Loading from ```PowerFactory```

In [ ]:
data_pf_fsm_smload = pd.read_csv(
    "./data/validation_oltc/sm_w_Zload400-800_fsm-validation_HV.csv",
    sep=";",
    dtype=float,
)
data_pf_fsm_smload.rename(
    columns={
        "Time in s": "t",
        "Voltage, Magnitude in p.u.": "Bus 0: voltage magnitude",
        "Voltage, Magnitude in p.u..1": "Bus 1: voltage magnitude",
    },
    inplace=True,
)
data_pf_fsm_smload.set_index("t", inplace=True)

data_pf_fsm_smload.head()

#### Comparing Against Each Other

In [ ]:
time = data_sm_load_fsm.index

# Calculate mean error
mean_errors_simple_fsm = np.zeros((2, len(time)))
for i in range(2):
    ext_voltages = data_pf_fsm_smload.iloc[:, i].values
    model_voltage = data_sm_load_fsm.iloc[:, i].values
    errors_fsm_simple = [
        abs(ext_v - model_v) for model_v, ext_v in zip(model_voltage, ext_voltages)
    ]
    mean_errors_simple_fsm[i] = errors_fsm_simple

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharex=True)

for i in range(2):
    axs[i].plot(
        data_pf_fsm_smload.index.values,
        data_pf_fsm_smload.iloc[:, i].values,
        color=ees_red,
        label="PowerFactory Data",
    )
    axs[i].plot(
        time,
        data_sm_load_fsm.iloc[:, i].values,
        linestyle="dashed",
        label="diffpssi Data",
    )
    axs[i].set_title(f"Bus {i}")
    axs[i].legend()
    axs[i].grid(True)

fig.supxlabel("Time in s")
fig.supylabel("Voltage Magnitude in p.u.")

plt.tight_layout()

# plt.savefig('./data/tds_comp_fsm_simple.pdf')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(time, mean_errors_simple_fsm[0], label="Error for Bus 0")
plt.plot(time, mean_errors_simple_fsm[1], label="Error for Bus 1")
plt.ylabel("Error in p.u.")
plt.xlabel("Time (s)")
plt.legend()
plt.grid(True)

plt.xlabel("Time in s")
plt.ylabel("Voltage Magnitude in p.u.")
# plt.tight_layout()

plt.savefig("./data/error_fsm_simple.pdf")

plt.show()